In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

CATALOGO = "desafio_vendas"
G, O = f"{CATALOGO}.gold", f"{CATALOGO}.ops"
spark.sql(f"CREATE VOLUME IF NOT EXISTS {G}.graficos")
PASTA = f"/Volumes/{CATALOGO}/gold/graficos"

v = spark.table(f"{G}.vw_vendas_analitica")
agg = spark.table(f"{G}.agg_vendas_mensal_produto")
rec = v.filter("flag_receita")          # receita = vendas aprovadas

plt.rcParams.update({"figure.figsize": (10, 4.5), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True})

def pdf(df):
    """Spark -> pandas, convertendo decimais para float."""
    for f in df.schema.fields:
        if isinstance(f.dataType, DecimalType):
            df = df.withColumn(f.name, F.col(f.name).cast("double"))
    return df.toPandas()

def brl(x, casas=0):
    return f"{x:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")

def mi(x, _=None):
    return f"R$ {brl(x / 1e6, 1)} mi"

def salvar(fig, nome):
    fig.tight_layout()
    fig.savefig(f"{PASTA}/{nome}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
k = pdf(rec.agg(
    F.count("*").alias("vendas"), F.sum("qtd_vendida").alias("unidades"),
    F.sum("valor_bruto").alias("bruta"), F.sum("valor_desconto").alias("desconto"),
    F.sum("valor_liquido").alias("liquida"), F.avg("valor_liquido").alias("ticket"),
    F.avg("valor_frete").alias("frete"), F.avg("avaliacao_produto").alias("nota"))).iloc[0]

t = pdf(v.agg(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("status_venda") == "Cancelada", 1).otherwise(0)).alias("canc"),
    F.sum(F.when(F.col("status_venda") == "Pendente", 1).otherwise(0)).alias("pend"),
    F.sum(F.when(F.col("status_venda") == "Desconhecido", 1).otherwise(0)).alias("desconhecidas"))).iloc[0]

indicadores = pd.DataFrame([
    ("Receita líquida (vendas aprovadas)", "R$ " + brl(k.liquida)),
    ("Receita bruta (aprovadas)", "R$ " + brl(k.bruta)),
    ("Desconto concedido (aprovadas)", f"R$ {brl(k.desconto)} ({brl(100 * k.desconto / k.bruta, 1)}% da receita bruta)"),
    ("Vendas aprovadas", brl(k.vendas)),
    ("Unidades vendidas (aprovadas)", brl(k.unidades)),
    ("Ticket médio (aprovadas)", "R$ " + brl(k.ticket, 2)),
    ("Frete médio (aprovadas)", "R$ " + brl(k.frete, 2)),
    ("Avaliação média (aprovadas)", brl(k.nota, 2)),
    ("Taxa de cancelamento", f"{brl(100 * t.canc / t.total, 1)}%"),
    ("Vendas pendentes", f"{brl(100 * t.pend / t.total, 1)}%"),
    ("Vendas com status desconhecido", f"{brl(100 * t.desconhecidas / t.total, 1)}%"),
], columns=["Indicador", "Valor"])
display(indicadores)

In [ ]:
m = pdf(agg.select("primeiro_dia_mes", "produto", "receita_liquida"))
m["primeiro_dia_mes"] = pd.to_datetime(m["primeiro_dia_mes"])
p = m.pivot(index="primeiro_dia_mes", columns="produto", values="receita_liquida").sort_index()

fig, ax = plt.subplots()
p.plot(ax=ax, marker="o", markersize=3)
ax.yaxis.set_major_formatter(FuncFormatter(mi))
ax.set_title("Receita líquida mensal por produto (vendas aprovadas)")
ax.set_xlabel(""); ax.set_ylim(bottom=0); ax.legend(title="")
salvar(fig, "01_receita_mensal_produto")

In [ ]:
r = pdf(rec.groupBy("produto").agg(F.sum("valor_liquido").alias("receita"),
                                   F.sum("qtd_vendida").alias("unidades")).orderBy(F.desc("receita")))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(r["produto"], r["receita"]); ax[0].yaxis.set_major_formatter(FuncFormatter(mi))
ax[0].set_title("Receita líquida por produto")
ax[1].bar(r["produto"], r["unidades"], color="tab:orange")
ax[1].yaxis.set_major_formatter(FuncFormatter(lambda x, _: brl(x)))
ax[1].set_title("Unidades vendidas por produto")
salvar(fig, "02_receita_unidades_produto")

In [ ]:
u = pdf(rec.filter("uf <> 'NI'").groupBy("regiao", "uf").agg(F.sum("valor_liquido").alias("receita")).orderBy("receita"))
cores = {"Norte": "tab:green", "Nordeste": "tab:orange", "Sudeste": "tab:blue", "Sul": "tab:red"}

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(u["uf"], u["receita"], color=[cores.get(x, "gray") for x in u["regiao"]])
ax.xaxis.set_major_formatter(FuncFormatter(mi))
ax.set_title("Receita líquida por UF (cor = região)")
ax.legend([plt.Rectangle((0, 0), 1, 1, color=c) for c in cores.values()], list(cores.keys()),
            title="Região", loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=4)
salvar(fig, "03_receita_uf_regiao")

In [ ]:
pg = pdf(rec.groupBy("tipo_pagamento").agg(F.sum("valor_liquido").alias("receita"),
                                           F.avg("valor_liquido").alias("ticket")).orderBy(F.desc("receita")))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(pg["tipo_pagamento"], pg["receita"]); ax[0].yaxis.set_major_formatter(FuncFormatter(mi))
ax[0].set_title("Receita líquida por forma de pagamento")
ax[1].bar(pg["tipo_pagamento"], pg["ticket"], color="tab:orange")
ax[1].yaxis.set_major_formatter(FuncFormatter(lambda x, _: "R$ " + brl(x)))
ax[1].set_title("Ticket médio por forma de pagamento")
salvar(fig, "04_pagamento")

In [ ]:
st = pdf(v.groupBy("produto", "status_venda").count())
pv = st.pivot(index="produto", columns="status_venda", values="count").fillna(0)
pv = pv.div(pv.sum(axis=1), axis=0) * 100
ordem = ["Aprovada", "Pendente", "Cancelada", "Desconhecido"]
cor = {"Aprovada": "tab:green", "Pendente": "tab:orange", "Cancelada": "tab:red", "Desconhecido": "gray"}
ordem = [c for c in ordem if c in pv.columns]

fig, ax = plt.subplots()
pv[ordem].plot(kind="barh", stacked=True, ax=ax, color=[cor[c] for c in ordem])
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:.0f}%"))
ax.set_title("Composição do status das vendas por produto"); ax.set_ylabel(""); ax.legend(title="", ncol=4, loc="lower center", bbox_to_anchor=(0.5, -0.25))
salvar(fig, "05_status_venda_produto")

In [ ]:
d = pdf(rec.groupBy("produto").agg(F.sum("valor_desconto").alias("desconto"), F.sum("valor_bruto").alias("bruta")))
d["pct"] = 100 * d["desconto"] / d["bruta"]

faixa = (F.when(F.col("desconto_pct") == 0, "0%").when(F.col("desconto_pct") <= 10, "1–10%")
          .when(F.col("desconto_pct") <= 20, "11–20%").when(F.col("desconto_pct") <= 30, "21–30%")
          .when(F.col("desconto_pct") <= 40, "31–40%").otherwise("41–50%"))
ordem_f = ["0%", "1–10%", "11–20%", "21–30%", "31–40%", "41–50%"]
fx = (pdf(v.filter("status_venda <> 'Cancelada'").groupBy(faixa.alias("faixa"))
          .agg(F.avg("qtd_vendida").alias("qtd_media"))).set_index("faixa").reindex(ordem_f))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(d["produto"], d["pct"]); ax[0].set_title("Desconto concedido sobre a receita bruta (%)")
ax[0].yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:.1f}%"))
ax[1].bar(fx.index, fx["qtd_media"], color="tab:orange"); ax[1].set_title("Unidades por venda, por faixa de desconto")
salvar(fig, "06_desconto")

In [ ]:
av = pdf(v.filter("avaliacao_produto IS NOT NULL AND status_venda <> 'Cancelada'")
          .groupBy("produto", "avaliacao_produto").count())
pv = av.pivot(index="avaliacao_produto", columns="produto", values="count").sort_index()

fig, ax = plt.subplots()
pv.plot(kind="bar", ax=ax, rot=0)
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: brl(x)))
ax.set_title("Distribuição das avaliações por produto (vendas não canceladas)")
ax.set_xlabel("Nota"); ax.legend(title="", loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3)
salvar(fig, "07_avaliacoes")

In [ ]:
pz = v.filter("status_venda <> 'Cancelada' AND NOT flag_retirada_inconsistente AND dias_ate_retirada IS NOT NULL")
h = pdf(pz.groupBy("dias_ate_retirada").count().orderBy("dias_ate_retirada"))
rg = pdf(pz.filter("regiao <> 'Não informada'").groupBy("regiao")
           .agg(F.avg("dias_ate_retirada").alias("dias")).orderBy("dias"))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(h["dias_ate_retirada"], h["count"]); ax[0].set_title("Distribuição do prazo até a retirada (dias)")
ax[1].barh(rg["regiao"], rg["dias"], color="tab:orange"); ax[1].set_title("Prazo médio por região (dias)")
salvar(fig, "08_prazo_retirada")

In [ ]:
# 1) Funil de registros
fn = pdf(spark.table(f"{O}.vw_funil_registros")).set_index("etapa")
sv = fn.loc["silver"]
rotulos = ["Bronze (lidos)", "Duplicadas removidas", "Quarentena (erros)", "Silver", "Gold (fato)"]
valores = [int(fn.loc["bronze", "qtd_gravada"]), int(sv["qtd_duplicada"]), int(sv["qtd_rejeitada"]),
           int(sv["qtd_gravada"]), int(fn.loc["gold", "qtd_gravada"])]
cores_f = ["tab:blue", "tab:gray", "tab:red", "tab:green", "tab:green"]

fig, ax = plt.subplots()
ax.barh(rotulos[::-1], valores[::-1], color=cores_f[::-1])
for i, val in enumerate(valores[::-1]):
    ax.text(val, i, f" {brl(val)}", va="center")
ax.set_title("Funil de registros: Bronze → Silver → Gold (última execução)")
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: brl(x)))
salvar(fig, "09_obs_funil")

# 2) Regras de qualidade com ocorrências
q = pdf(spark.table(f"{O}.vw_qualidade_ultima").filter("qtd_afetada > 0")).sort_values("pct_afetado")
cor_s = {"QUARENTENA": "tab:red", "ALERTA": "tab:orange", "CRITICO": "black"}
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(q["regra_id"] + " · " + q["descricao"].str.slice(0, 55), q["pct_afetado"], color=q["situacao"].map(cor_s))
ax.set_title("Regras de qualidade com ocorrências (% dos registros avaliados)")
ax.legend([plt.Rectangle((0, 0), 1, 1, color=c) for c in cor_s.values()], list(cor_s.keys()), loc="lower right")
salvar(fig, "10_obs_qualidade")

# 3) Duração por etapa nas últimas execuções
du = pdf(spark.table(f"{O}.log_execucao").select("id_execucao", "etapa", "inicio", "duracao_s"))
du["inicio"] = pd.to_datetime(du["inicio"])
ult = du.groupby("id_execucao")["inicio"].min().sort_values().index[-10:]
pvd = (du[du["id_execucao"].isin(ult)].pivot_table(index="id_execucao", columns="etapa", values="duracao_s", aggfunc="sum")
         .reindex(ult))
pvd.index = [f"exec {i + 1}" for i in range(len(pvd))]
fig, ax = plt.subplots()
pvd[[c for c in ["bronze", "silver", "gold", "previsao"] if c in pvd.columns]].plot(kind="bar", stacked=True, ax=ax, rot=0)
ax.set_title("Duração por etapa nas últimas execuções (segundos)"); ax.set_xlabel(""); ax.legend(title="")
salvar(fig, "11_obs_duracao")

print("PNGs salvos em:", PASTA)